[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nattaponm/Teach_stormTracks_KKN_May2024_HID_doppler/blob/main/notebooks/03_tdating_tracks_dt15_10_5.ipynb)


# บทที่ 03 — ติดตามพายุด้วย T-DaTing: ทดลอง Δt = 15, 10 และ 5 นาที

เรดาร์ KKN ให้ภาพทุก 15 นาที พายุเคลื่อนที่ 20–40 km/h จึงเลื่อนไป 5–10 km ระหว่างภาพ ซึ่งเทียบได้กับขนาดของเซลล์เอง การจับคู่ "ลูกเดิม" ข้ามเวลาจึงผิดพลาดได้ บทนี้ทดลองสร้างภาพ 10 และ 5 นาทีด้วย advection interpolation แล้วดูว่าเส้นทางที่ได้ต่างกันอย่างไร — ต่อยอดจาก repo `Teach_Tdating_strom_tracks_HID_CloudProperties` ซึ่งทำวันเดียว มาเป็นทั้งเดือน

### วัตถุประสงค์การเรียนรู้
1. อธิบายขั้นตอนของ T-DaTing (ระบุเซลล์ → คาดตำแหน่ง → จับคู่) และความหมายของพารามิเตอร์หลัก
2. สร้างภาพระหว่างเวลาด้วย advection interpolation แบบ linZ และอธิบายว่าทำไมภาพสังเคราะห์ใช้ "ติดตาม" ได้แต่ไม่ใช้ "วัดความเข้ม"
3. เปรียบเทียบจำนวน อายุ ความเร็ว และความยาวเส้นทางระหว่าง Δt ทั้งเดือน
4. จับคู่เส้นทางข้ามความละเอียดด้วย IoU ของ footprint
5. อธิบายผลของข้อมูลขาด (segment) และเส้นทางที่ถูกตัด (censored)

In [ ]:
# @title ⚙️ เตรียมสภาพแวดล้อม (รันก่อนเสมอ)
import os, sys, subprocess
REPO = "Teach_stormTracks_KKN_May2024_HID_doppler"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/nattaponm/{REPO}.git", f"/content/{REPO}"], check=True)
    else:
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"])
    os.chdir(f"/content/{REPO}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pysteps==1.21.5", "opencv-python-headless"], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "prep"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("โฟลเดอร์ทำงาน:", os.getcwd(), "| CPU", os.cpu_count(), "core")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats, ndimage
warnings.filterwarnings("ignore")

from utils.config import load_settings, repo_path
from utils import maps as M, geo as G, data as D, analysis as A, track_tools as tt
s = load_settings()
M.setup_style()
pd.set_option("display.width", 200, "display.max_columns", 30)
DEM = G.load_dem(s)                                   # (array, extent) — สร้างจาก Copernicus ครั้งแรกถ้ายังไม่มี
idx = D.day_index()
KEY_DAYS = list(idx.loc[idx["in_repo"], "day"]) if len(idx) else [pd.Timestamp("2024-05-03")]
KEY = KEY_DAYS[0]                                     # วันหลักที่มีกริดใน repo (เปลี่ยนเป็นวันอื่นได้ จะดาวน์โหลดจาก Zenodo)
print("วันหลักใน repo:", [f"{d:%d %b}" for d in KEY_DAYS], "| ใช้วัน:", f"{KEY:%d %b %Y}")

try:
    from pysteps.tracking import tdating  # noqa: F401
    HAS_PYSTEPS = True
except Exception:
    HAS_PYSTEPS = False
print("pysteps:", HAS_PYSTEPS, "| พารามิเตอร์ T-DaTing:", tt.PARAMS, tt.TRACK_KW)

## 1. T-DaTing ทำงานอย่างไร

**T-DaTing** (Thunderstorm Detection and Tracking; Feldmann et al. 2021) ใน pysteps:

1. **ระบุเซลล์:** pixel ที่ $Z \ge$ `minref` (33 dBZ) แยกเป็นเซลล์ด้วย watershed เมื่อสองยอดต่างจากจุดต่ำระหว่างกันอย่างน้อย `mindiff` (5 dB) เซลล์ต้องมีขนาด ≥ `minsize` (15 km²) และมีค่าสูงสุด ≥ `minmax` (40 dBZ) ค่าที่เกิน `maxref` (45 dBZ) ถูกตัดก่อน watershed เพื่อไม่ให้แกนแรงแตกย่อยเกินไป
2. **คาดตำแหน่ง:** เลื่อนเซลล์เวลา $t-1$ ไปตามความเร็วที่ประมาณได้
3. **จับคู่:** เซลล์ที่ $t$ ซึ่งซ้อนทับเซลล์ที่เลื่อนแล้วมากพอ (`match_frac` = 0.25) ถือเป็นลูกเดียวกัน ส่วน `mindis` (8 km) จำกัดระยะของการจับคู่

**ภาพสังเคราะห์ (advection interpolation)** ระหว่างภาพจริง $Z_a$ ($t_0$) กับ $Z_b$ ($t_0+\Delta t$) ที่สัดส่วนเวลา $w$:

$$Z(t_0 + w\Delta t) = 10\log_{10}\!\Big[(1-w)\,10^{Z_a'(w)/10} + w\,10^{Z_b'(w)/10}\Big]$$

โดย $Z_a'$ คือ $Z_a$ ที่เลื่อนไปข้างหน้า $w$ ช่วง และ $Z_b'$ คือ $Z_b$ ที่เลื่อนถอยหลัง $1-w$ ช่วง ตามสนามการเคลื่อนที่ Lucas–Kanade ผลคือภาพ **เรียบ** แต่ **แกน ≥ 45 dBZ อ่อนลง** (ถูกเฉลี่ย) — ค่า HID/dual-pol/Doppler ทั้งหมดในบทต่อไปจึงดึงจากภาพจริงเท่านั้น

## 2. ทดลองสด: ติดตาม 2 ชั่วโมงรอบช่วงพีคของวันหลัก (15 vs 5 นาที)

In [ ]:
Z, meta = D.load_cappi(KEY)
times = pd.to_datetime(meta["timestamps"])
area = np.array([np.nansum(z >= 35) for z in Z]); it = int(np.nanargmax(area))
i0, i1 = max(0, it - 4), min(len(Z) - 1, it + 4)
assert meta["available"][i0:i1 + 1].all(), "ช่วงนี้มีข้อมูลขาด — เลือกช่วงอื่น"
Zs, ts = Z[i0:i1 + 1], list(times[i0:i1 + 1].to_pydatetime())
print(f"ช่วง {ts[0]:%H:%M}–{ts[-1]:%H:%M} UTC ({len(ts)} ภาพจริง)")
if HAS_PYSTEPS:
    tr15 = tt.run_tdating(Zs, ts)
    Z5, t5, obs5 = tt.advect_interpolate(Zs, ts, n_sub=3, method="linZ")
    tr5 = tt.run_tdating(Z5, t5)
    a45 = [np.nansum(z >= 45) for z in Z5]
    print(f"15 นาที: {len(tr15)} เส้นทาง | 5 นาที: {len(tr5)} เส้นทาง จาก {len(t5)} ภาพ ({(~obs5).sum()} ภาพสังเคราะห์)")
    print(f"พื้นที่ ≥ 45 dBZ มัธยฐาน: ภาพจริง {np.median(np.array(a45)[obs5]):.0f} km², ภาพสังเคราะห์ {np.median(np.array(a45)[~obs5]):.0f} km²")
else:
    print("ไม่มี pysteps — ข้ามการทดลองสด ใช้ผลที่คำนวณไว้แล้วในหัวข้อถัดไป")

In [ ]:
if HAS_PYSTEPS:
    fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.6), layout="constrained")
    for ax, (tr, ttl) in zip(axs, [(tr15, "Δt = 15 นาที (ภาพจริง)"), (tr5, "Δt = 5 นาที (linZ)")]):
        A.show_grid(ax, np.where(Zs[it - i0] >= 20, Zs[it - i0], np.nan), meta, cmap="Greys", vmin=20, vmax=60, alpha=0.5)
        for k, c in enumerate(tr):
            ax.plot(meta["x"][c["cen_x"].round().astype(int)] / 1e3, meta["y"][c["cen_y"].round().astype(int)] / 1e3,   # cen_x/cen_y = index คอลัมน์/แถว
                    "-", lw=1, color=plt.get_cmap("tab10")(k % 10))
        M.add_range_rings(ax, s, rings_km=[120, 240], label=False, name=False)
        ax.set_xlim(-245, 245); ax.set_ylim(-245, 245); ax.set_aspect("equal"); ax.set_title(f"{ttl}: {len(tr)} เส้นทาง", loc="left")
    M.savefig(fig, "nb03_live_tracks")
    plt.show()

## 3. ผลทั้งเดือน (คำนวณไว้แล้วใน prep/P3)

P3 ติดตามทุกวันเป็น **segment** (ช่วงที่ภาพจริงต่อเนื่องทุก 15 นาที) ช่องเวลาที่ขาดตัดเส้นทาง และภาพ 5/10 นาทีสร้างเฉพาะภายใน segment เส้นทางที่เริ่มหรือจบที่ขอบ segment ติดป้าย **censored** เพราะอายุจริงอาจยาวกว่าที่วัดได้

In [ ]:
summ = D.table("tracking_summary.csv"); seg = D.table("segments.csv")
print(f"segment ทั้งหมด {len(seg)} ใน {seg['day'].nunique()} วัน")
summ

In [ ]:
TAGS = ["15min", "10min_linZ", "05min_linZ"]
TT = pd.concat([A.month_tracks(t, min_life=0).assign(tag=t) for t in TAGS], ignore_index=True)
TT["tag"] = pd.Categorical(TT["tag"], TAGS, ordered=True)
long_ = TT[TT["lifetime_min"] >= 30]
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.8), layout="constrained")
cols = ["#1b9e77", "#d95f02", "#7570b3"]
for ax, (v, lab) in zip(axs, [("lifetime_min", "อายุ (นาที)"), ("speed_kmh", "ความเร็ว (km/h)"), ("path_km", "ความยาวเส้นทาง (km)")]):
    tt.boxplot_groups(ax, long_, v, "tag", order=TAGS, colors=cols, ylabel=lab, points=False)
    ax.tick_params(axis="x", labelsize=6)
fig.suptitle("เส้นทางอายุ ≥ 30 นาที แยกตาม Δt", fontsize=9)
M.savefig(fig, "nb03_track_properties_by_dt")
plt.show()

**อ่านผล:** Δt สั้นลงมักได้ **เส้นทางมากขึ้นและยาวขึ้น** เพราะการเคลื่อนที่ระหว่างภาพเล็กลง การจับคู่ด้วยการซ้อนทับจึงสำเร็จบ่อยขึ้น แต่ภาพสังเคราะห์อาจ "เชื่อม" เซลล์ที่ไม่ใช่ลูกเดียวกันได้ด้วย คำถามคือเส้นทางที่เพิ่มมาเป็นของจริงหรือไม่ — หัวข้อถัดไปตอบด้วยการจับคู่ footprint

## 4. จับคู่เส้นทางข้าม Δt ด้วย IoU ของ footprint

footprint = ชุด pixel ทั้งหมดที่เส้นทางเคยกวาดผ่าน

$$\text{IoU}(A,B) = \frac{|A\cap B|}{|A\cup B|}$$

เส้นทาง 15 นาทีหนึ่งเส้นจับคู่กับเส้นทาง 5 นาทีที่ IoU สูงสุด (≥ 0.1)

In [ ]:
lab15, t15_, _ = D.load_labels(KEY, "15min")
lab05, t05_, _ = D.load_labels(KEY, "05min_linZ")
fp15, fp05 = tt.footprint_masks(lab15), tt.footprint_masks(lab05)
m = tt.match_tracks(fp15, fp05, min_iou=0.1)
T15k = A.month_tracks("15min", 0); T15k = T15k[T15k["day"] == KEY].set_index("track_id")
m = m.join(T15k[["lifetime_min"]])
print(f"{KEY:%d %b}: เส้นทาง 15 นาที {len(m)} เส้น จับคู่ได้ {m['match'].notna().sum()} ({100 * m['match'].notna().mean():.0f}%) | IoU มัธยฐาน {m['iou'].median():.2f}")
print(m.groupby(pd.cut(m["lifetime_min"], [0, 30, 60, 1e4], right=False), observed=True)["iou"].median().rename("IoU มัธยฐานตามอายุ").to_string())

## 5. แผนที่เส้นทางของวันหลัก 3 ความละเอียด

In [ ]:
CC = {t: A.month_cells(t) for t in TAGS}
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.9), layout="constrained")
for ax, t in zip(axs, TAGS):
    C = CC[t]; C = C[C["day"] == KEY]
    life = C.groupby("uid")["time"].agg(lambda x: (x.max() - x.min()).total_seconds() / 60)
    M.add_hillshade(ax, *DEM, alpha=0.5)
    for uid, g in C[C["uid"].isin(life.index[life >= 30])].groupby("uid"):
        g = g.sort_values("time")
        sc = ax.scatter(g["cx"] / 1e3, g["cy"] / 1e3, c=(g["time"].dt.hour + g["time"].dt.minute / 60 + 7) % 24, cmap="plasma",
                        vmin=12, vmax=24, s=2, zorder=5)
        ax.plot(g["cx"] / 1e3, g["cy"] / 1e3, color="k", lw=0.4, zorder=4)
    M.add_range_rings(ax, s, rings_km=[120, 240], label=False, name=False)
    ax.set_xlim(-245, 245); ax.set_ylim(-245, 245); ax.set_aspect("equal"); ax.tick_params(labelsize=6)
    ax.set_title(f"{t} ({(life >= 30).sum()} เส้นทาง)", loc="left", fontsize=8)
fig.colorbar(sc, ax=axs, shrink=0.7, label="เวลาท้องถิ่น (น.)")
fig.suptitle(f"เส้นทางอายุ ≥ 30 นาที {KEY:%d %b %Y}", fontsize=9)
M.savefig(fig, "nb03_tracks_three_dt")
plt.show()

## 6. แบบฝึกหัด

**ข้อ 1** นับเส้นทางอายุ ≥ 60 นาทีแยกตาม Δt และช่วงของเดือน (`phase`) Δt ใดให้เส้นทางยาวมากที่สุด และแตกต่างกันในแต่ละช่วงหรือไม่

In [ ]:
# เขียนโค้ดของตนเองที่นี่

In [ ]:
# @title เฉลยข้อ 1
TT[TT["lifetime_min"] >= 60].pivot_table(index="phase", columns="tag", values="uid", aggfunc="count", observed=False).fillna(0).astype(int)

**ข้อ 2** ทดสอบว่าความเร็วของเส้นทาง (≥ 30 นาที) ต่างกันระหว่าง Δt หรือไม่ ด้วย Kruskal–Wallis ระวัง: เส้นทางชุดต่าง Δt **เป็นพายุชุดเดียวกัน** การทดสอบนี้จึงละเมิดสมมติฐานความเป็นอิสระ ลองอธิบายว่าควรทดสอบแบบจับคู่ (paired) อย่างไร

In [ ]:
# เขียนโค้ดของตนเองที่นี่

In [ ]:
# @title เฉลยข้อ 2
print(stats.kruskal(*[long_.loc[long_["tag"] == t, "speed_kmh"].dropna() for t in TAGS]))
# แบบจับคู่: ใช้ผล match_tracks (หัวข้อ 4) จับคู่เส้นทาง 15 ↔ 5 นาทีของพายุลูกเดียวกัน
# แล้วทดสอบผลต่างความเร็วรายคู่ด้วย Wilcoxon signed-rank
T05k = A.month_tracks("05min_linZ", 0); T05k = T05k[T05k["day"] == KEY].set_index("track_id")
pr = m.dropna(subset=["match"]).copy()
pr["v15"] = T15k.loc[pr.index, "speed_kmh"].values
pr["v05"] = T05k.loc[pr["match"].astype(int), "speed_kmh"].values
pr = pr.dropna(subset=["v15", "v05"])
if len(pr) >= 6:
    print("Wilcoxon (จับคู่):", stats.wilcoxon(pr["v15"], pr["v05"]))

**ข้อ 3** สัดส่วนเส้นทางที่ censored ต่างกันระหว่างวันหรือไม่ วันที่มี segment มากกว่า 1 ช่วง (ข้อมูลขาดกลางวัน) มีสัดส่วน censored สูงกว่าหรือไม่

In [ ]:
# เขียนโค้ดของตนเองที่นี่

In [ ]:
# @title เฉลยข้อ 3
T15 = A.month_tracks("15min", 0)
T15["censored"] = T15["censored_start"] | T15["censored_end"]
c = T15.groupby("day").agg(censored_pct=("censored", "mean"), n=("uid", "size"))
c["n_segments"] = seg.groupby("day").size().reindex(c.index).fillna(0).astype(int)
c["censored_pct"] *= 100
c.round(1)

## 7. คำถามชวนคิด
1. ถ้าเรดาร์สแกนทุก 5 นาทีจริง ผลการติดตามจะต่างจากภาพสังเคราะห์ 5 นาทีอย่างไร การทดสอบ leave-one-out (`tt.loo_interpolation_test`) ช่วยตอบได้แค่ไหน
2. เส้นทางที่ "ยาวขึ้น" เมื่อใช้ Δt สั้น อาจเกิดจากการเชื่อมสองพายุเข้าด้วยกันผิด ๆ จะออกแบบการตรวจสอบอย่างไร (เช่น ดูการกระโดดของ centroid หรือความเร็วที่ผิดธรรมชาติ)
3. พารามิเตอร์ T-DaTing ปรับมาจากวันที่ 3 พ.ค. (พายุรุนแรง) จะเหมาะกับพายุช่วงฤดูฝนที่แผ่กว้างและอ่อนกว่าหรือไม่

## 8. แนวทางต่อยอด
- ทดสอบความไวของพารามิเตอร์ (`minref`, `mindiff`, `match_frac`) แบบ grid search และวัดเสถียรภาพของสถิติเส้นทาง
- ใช้วิธีติดตามอื่นเทียบ เช่น TINT/TITAN หรือ tobac
- ใช้ survival analysis (Kaplan–Meier) กับอายุเส้นทาง เพื่อจัดการเส้นทาง censored อย่างถูกต้อง

**อ้างอิง:** Feldmann et al. (2021) *Wea. Forecasting* 36, 1599–1622; Germann & Zawadzki (2002) *MWR* 130, 2859–2873; Pulkkinen et al. (2019) *GMD* 12, 4185–4219